In [ ]:
######################## This script was used to generate the Squidiff results   ########
import sys
import os

import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import torch
torch.cuda.set_device(2)
print(f"当前设备: {torch.cuda.current_device()}")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  ## 避免碎片化
torch.cuda.empty_cache()   # 先清除缓存
import matplotlib.font_manager
from matplotlib import rcParams
font_list = []
fpaths = matplotlib.font_manager.findSystemFonts()
for i in fpaths:
    try:
        f = matplotlib.font_manager.get_font(i)
        font_list.append(f.family_name)
    except RuntimeError:
        pass

font_list = set(font_list)
plot_font = 'Helvetica' if 'Helvetica' in font_list else 'FreeSans'
rcParams['font.family'] = plot_font
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'figure.figsize': (3,3)})
rcParams.update({'savefig.dpi': 500})

%load_ext autoreload
%autoreload 2

sys.path.append('../')
os.chdir('external_inputs/Squidiff')
import sample_squidiff

# import plot_util 
os.chdir('external_inputs/wangyuting')
import warnings
warnings.filterwarnings('ignore')


In [ ]:
sampler = sample_squidiff.sampler(
    model_path = 'external_inputs/model.pt',#'diff_results/model.pt',
    gene_size = 3253,
    output_dim = 3253,
    use_drug_structure = False
)
torch.cuda.empty_cache()


In [ ]:
train_adata = sc.read_h5ad('external_inputs/diff_train.h5ad')  # 
z_sem = sampler.model.encoder(
    torch.tensor(train_adata.X).to('cuda')
)
torch.cuda.empty_cache()
scrnas_pred = sampler.pred(z_sem, gene_size = train_adata.shape[1])
torch.cuda.empty_cache()
x_T = sampler.stochastic_encode(
    sampler.model,
    x = torch.tensor(train_adata.X, dtype = torch.float32).to('cuda'),
    t = 199,
    model_kwargs={'x_start':torch.tensor(train_adata.X, dtype = torch.float32).to('cuda'),
                  'z_mod':z_sem,
    }
)['sample']


In [ ]:
train_adata.shape


In [ ]:
# sc.tl.umap()

In [ ]:
####### 画 pca 的图
import numpy as np
import torch
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# 1. 数据准备
# 假设 z_sem 是你的 tensor，先转到 CPU 并转为 numpy 数组
z_sem_np = z_sem.detach().cpu().numpy()

# 2. 执行 PCA 降维到 2 维
pca = PCA(n_components=2, random_state=42)
z_pca = pca.fit_transform(z_sem_np)

# 3. 获取分组标签
labels = train_adata.obs['Group'].values

# 4. 绘图
plt.figure(figsize=(10, 9))

# 定义颜色和点大小，与图中效果匹配
palette = ['#1a5fff', '#ff2b56', '#ffd700','green']  # 蓝、红、黄
sns.scatterplot(
    x=z_pca[:, 0],
    y=z_pca[:, 1],
    hue=labels,
    palette=palette,
    s=80,  # 点大小
    alpha=0.8,
    edgecolor='none'
)

# 美化绘图
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of z_sem embeddings')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(title='Group', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
# 显示或保存图片
plt.show()
# plt.savefig('pca_z_sem.png', dpi=300, bbox_inches='tight')


In [ ]:
####### 画 pca 的图, x_T 
import numpy as np
import torch
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# 1. 数据准备
# 假设 z_sem 是你的 tensor，先转到 CPU 并转为 numpy 数组
z_sem_np = x_T.detach().cpu().numpy()

# 2. 执行 PCA 降维到 2 维
pca = PCA(n_components=2, random_state=42)
z_pca = pca.fit_transform(z_sem_np)

# 3. 获取分组标签
labels = train_adata.obs['Group'].values

# 4. 绘图
plt.figure(figsize=(10, 9))

# 定义颜色和点大小，与图中效果匹配
palette = ['#1a5fff', '#ff2b56', '#ffd700','green']  # 蓝、红、黄
sns.scatterplot(
    x=z_pca[:, 0],
    y=z_pca[:, 1],
    hue=labels,
    palette=palette,
    s=80,  # 点大小
    alpha=0.8,
    edgecolor='none'
)

# 美化绘图
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of x_T embeddings')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(title='Group', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

# 显示或保存图片
plt.show()
# plt.savefig('pca_z_sem.png', dpi=300, bbox_inches='tight')


In [ ]:
# 直接使用 AnnData 中已有的 PCA 结果
scatter_x = train_adata.obsm['X_pca'][:, 0]
scatter_y = train_adata.obsm['X_pca'][:, 1]

plt.figure(figsize=(10, 9))
sns.scatterplot(
    x=scatter_x,
    y=scatter_y,
    hue=train_adata.obs['Group'],
    palette=['#1a5fff', '#ff2b56', '#ffd700','green'],
    s=80,
    alpha=0.8,
    edgecolor='none'
)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of adata embeddings')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(title='Group', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## predict differention/tragectory of different Fibroblast subtypes

In [ ]:
torch.cuda.empty_cache()  
# get the day 0 vector from test data
z_sem = sampler.model.encoder(torch.tensor(train_adata.X).to('cuda'))
# get the represented differentiation latent vector  
direct_z_sem = z_sem[train_adata.obs.Group=='eFibro'].mean(axis=0) - z_sem[train_adata.obs.Group=='qFibro'].mean(axis=0)


In [ ]:
test_adata = sc.read_h5ad('external_inputs/test_adata.h5ad')
# sc.pp.highly_variable_genes(test_adata, n_top_genes=8000)


In [ ]:
var_names = list(train_adata.var_names)   ### pd.read_csv('external_inputs/train_adata_hvg.tsv',sep='\t')   # test_adata.var.highly_variable
# var_names = var_names['hvg'].to_list()
test_adata_hvg = test_adata[:, var_names ]


In [ ]:
test_adata_hvg

In [ ]:
z_sem_test_day0 = sampler.model.encoder(torch.tensor(test_adata_hvg[test_adata_hvg.obs['Group']=='qFibro'].X).to('cuda'))
z_sem_test_day1 = sampler.model.encoder(torch.tensor(test_adata_hvg[test_adata_hvg.obs['Group']=='iFibro'].X).to('cuda'))
z_sem_test_day2 = sampler.model.encoder(torch.tensor(test_adata_hvg[test_adata_hvg.obs['Group']=='Myfibro'].X).to('cuda'))
z_sem_test_day3 = sampler.model.encoder(torch.tensor(test_adata_hvg[test_adata_hvg.obs['Group']=='eFibro'].X).to('cuda'))


In [ ]:
r_all = []
pearsonr_all = []

for i in range(6):
    sample_interp0 = sampler.pred(z_sem = z_sem_test_day0,
                                  gene_size = 3253,
                                 )
    sample_interp1 = sampler.pred(z_sem = z_sem_test_day1,
                                  gene_size = 3253,
                                 )
    sample_interp2 = sampler.pred(z_sem = z_sem_test_day2,
                                  gene_size = 3253,
                                 )
    sample_interp3 = sampler.pred(z_sem = z_sem_test_day3,
                                  gene_size = 3253,
                                 )
    # sample_interp1 = sampler.interp_with_direction(z_sem_origin = z_sem_test_day0,
    #                                                gene_size = 8000,
    #                                                direction = direct_z_sem,
    #                                                scale = 1/3
    #                                               )
    # sample_interp2 = sampler.interp_with_direction(z_sem_origin = z_sem_test_day0,
    #                                                gene_size = 8000,
    #                                                direction = direct_z_sem,
    #                                                scale = 2/3
    #                                               )
    # sample_interp3 = sampler.pred(z_sem = z_sem_test_day3,
    #                               gene_size = 8000,
    #                              )
    r2_, pearsonr_ = sampler.cal_metric(sample_interp0, test_adata_hvg[test_adata_hvg.obs['Group']=='qFibro'])
    r_all.append(['squidiff',i,'qFibro',r2_])
    pearsonr_all.append(['squidiff',i,'qFibro',pearsonr_])
    r2_, pearsonr_ = sampler.cal_metric(sample_interp1, test_adata_hvg[test_adata_hvg.obs['Group']=='iFibro'])   # day1
    r_all.append(['squidiff',i,'iFibro',r2_])
    pearsonr_all.append(['squidiff',i,'iFibro',pearsonr_])
    r2_, pearsonr_ = sampler.cal_metric(sample_interp2,  test_adata_hvg[test_adata_hvg.obs['Group']=='Myfibro'])     # day 2
    r_all.append(['squidiff',i,'Myfibro',r2_])
    pearsonr_all.append(['squidiff',i,'Myfibro',pearsonr_])
    r2_, pearsonr_ = sampler.cal_metric(sample_interp3, test_adata_hvg[test_adata_hvg.obs['Group']=='eFibro'])   # day3
    r_all.append(['squidiff',i,'eFibro',r2_])
    pearsonr_all.append(['squidiff',i,'eFibro',pearsonr_])
    
#pd.DataFrame(r_all).to_csv('/hpc/mydata/siyu.he/Siyu_projects/Squidiff_reproducibility/fig2_diff_r_all_new.csv')
#pd.DataFrame(pearsonr_all).to_csv('/hpc/mydata/siyu.he/Siyu_projects/Squidiff_reproducibility/fig2_diff_pearsonr_all_new.csv')


In [ ]:
torch.cuda.empty_cache()


In [ ]:
pearsonr_all


In [ ]:
sample_interp2.detach().cpu().median(axis=0).values


In [ ]:
# sorted(dir())


In [ ]:
len(test_adata_hvg[test_adata_hvg.obs['Group']=='Myfibro'].X.mean(axis=0))


In [ ]:
plt.subplots(1,1,figsize=(2,2),dpi=800)
plt.scatter(sample_interp0.detach().cpu().median(axis=0).values,       # sample_interp2.detach().cpu().mean(axis=0)
            test_adata_hvg[test_adata_hvg.obs['Group']=='qFibro'].X.mean(axis=0),
            s=4,
            c='#c29523'
           )
plt.xlim([0,60])
plt.ylim([0,10])

sampler.cal_metric(sample_interp0, test_adata_hvg[test_adata_hvg.obs['Group']=='qFibro'])


In [ ]:
plt.subplots(1,1,figsize=(2,2),dpi=800)
plt.scatter(sample_interp1.detach().cpu().median(axis=0).values,       # sample_interp2.detach().cpu().mean(axis=0)
            test_adata_hvg[test_adata_hvg.obs['Group']=='iFibro'].X.mean(axis=0),
            s=4,
            c='#c29523'
           )
plt.xlim([0,60])
plt.ylim([0,10])

sampler.cal_metric(sample_interp1, test_adata_hvg[test_adata_hvg.obs['Group']=='iFibro'])


In [ ]:
plt.subplots(1,1,figsize=(2,2),dpi=800)
plt.scatter(sample_interp2.detach().cpu().median(axis=0).values,       # sample_interp2.detach().cpu().mean(axis=0)
            test_adata_hvg[test_adata_hvg.obs['Group']=='Myfibro'].X.mean(axis=0),
            s=4,
            c='#c29523'
           )
plt.xlim([0,60])
plt.ylim([0,10])

sampler.cal_metric(sample_interp2, test_adata_hvg[test_adata_hvg.obs['Group']=='Myfibro'])


In [ ]:
plt.subplots(1,1,figsize=(2,2),dpi=800)
plt.scatter(sample_interp3.detach().cpu().median(axis=0).values,       # sample_interp2.detach().cpu().mean(axis=0)
            test_adata_hvg[test_adata_hvg.obs['Group']=='eFibro'].X.mean(axis=0),
            s=4,
            c='#c29523'
           )
plt.xlim([0,60])
plt.ylim([0,10])

sampler.cal_metric(sample_interp3, test_adata_hvg[test_adata_hvg.obs['Group']=='eFibro'])


In [ ]:
####### test 的数据似乎效果都不好，试下 train_adata 的
z_sem_test_day0 = sampler.model.encoder(torch.tensor(train_adata[train_adata.obs['Group']=='qFibro'].X).to('cuda'))
z_sem_test_day1 = sampler.model.encoder(torch.tensor(train_adata[train_adata.obs['Group']=='iFibro'].X).to('cuda'))
z_sem_test_day2 = sampler.model.encoder(torch.tensor(train_adata[train_adata.obs['Group']=='Myfibro'].X).to('cuda'))
z_sem_test_day3 = sampler.model.encoder(torch.tensor(train_adata[train_adata.obs['Group']=='eFibro'].X).to('cuda'))

r_all = []
pearsonr_all = []

for i in range(6):
    sample_interp0 = sampler.pred(z_sem = z_sem_test_day0,
                                  gene_size = 3253,
                                 )
    sample_interp1 = sampler.pred(z_sem = z_sem_test_day1,
                                  gene_size = 3253,
                                 )
    sample_interp2 = sampler.pred(z_sem = z_sem_test_day2,
                                  gene_size = 3253,
                                 )
    sample_interp3 = sampler.pred(z_sem = z_sem_test_day3,
                                  gene_size = 3253,
                                 )
    # sample_interp1 = sampler.interp_with_direction(z_sem_origin = z_sem_test_day0,
    #                                                gene_size = 8000,
    #                                                direction = direct_z_sem,
    #                                                scale = 1/3
    #                                               )
    # sample_interp2 = sampler.interp_with_direction(z_sem_origin = z_sem_test_day0,
    #                                                gene_size = 8000,
    #                                                direction = direct_z_sem,
    #                                                scale = 2/3
    #                                               )
    # sample_interp3 = sampler.pred(z_sem = z_sem_test_day3,
    #                               gene_size = 8000,
    #                              )
    r2_, pearsonr_ = sampler.cal_metric(sample_interp0,train_adata[train_adata.obs['Group']=='qFibro'])
    r_all.append(['squidiff',i,'qFibro',r2_])
    pearsonr_all.append(['squidiff',i,'qFibro',pearsonr_])
    r2_, pearsonr_ = sampler.cal_metric(sample_interp1, train_adata[train_adata.obs['Group']=='iFibro'])   # day1
    r_all.append(['squidiff',i,'iFibro',r2_])
    pearsonr_all.append(['squidiff',i,'iFibro',pearsonr_])
    r2_, pearsonr_ = sampler.cal_metric(sample_interp2,  train_adata[train_adata.obs['Group']=='Myfibro'])     # day 2
    r_all.append(['squidiff',i,'Myfibro',r2_])
    pearsonr_all.append(['squidiff',i,'Myfibro',pearsonr_])
    r2_, pearsonr_ = sampler.cal_metric(sample_interp3, train_adata[train_adata.obs['Group']=='eFibro'])   # day3
    r_all.append(['squidiff',i,'eFibro',r2_])
    pearsonr_all.append(['squidiff',i,'eFibro',pearsonr_])
    
#pd.DataFrame(r_all).to_csv('/hpc/mydata/siyu.he/Siyu_projects/Squidiff_reproducibility/fig2_diff_r_all_new.csv')
#pd.DataFrame(pearsonr_all).to_csv('/hpc/mydata/siyu.he/Siyu_projects/Squidiff_reproducibility/fig2_diff_pearsonr_all_new.csv')

torch.cuda.empty_cache()    ## 是不是feature数太多了，导致效果这么差啊


In [ ]:
z_sem_test_day0[0]      ### 


In [ ]:
sample_interp0[0].median()

In [ ]:
sample_interp0[0].quantile(0.5, dim=0)


In [ ]:
train_adata.X.max()

In [ ]:
sample_interp0.detach().cpu().nanmedian(axis=0).values


In [ ]:
plt.subplots(1,1,figsize=(2,2),dpi=800)
plt.scatter(sample_interp0.detach().cpu().nanmedian(axis=0).values ,        # .median(axis=0).values,       # sample_interp2.detach().cpu().mean(axis=0)
            train_adata[train_adata.obs['Group']=='qFibro'].X.mean(axis=0),
            s=4,
            c='#c29523'
           )
plt.xlim([0,60])
plt.ylim([0,10])

sampler.cal_metric(sample_interp0, train_adata[train_adata.obs['Group']=='qFibro'])


In [ ]:
train_adata.obs['Group'].value_counts()


In [ ]:
train_adata.shape


In [ ]:
embedding_combined[0]


In [ ]:
colorlist

In [ ]:
# Combine all points
from sklearn.decomposition import PCA
import umap
combined_points = np.concatenate([sample_interp0.detach().cpu().numpy(),
                                  sample_interp1.detach().cpu().numpy(),
                                  sample_interp2.detach().cpu().numpy(),
                                  sample_interp3.detach().cpu().numpy(),
                                 ], axis=0)
combined_labels  = np.array(['qFibro']*463+['iFibro']*1544+['Myfibro']*1483+['eFibro']*1200)  ## 按照 subtype 的细胞数来

pca = PCA(n_components=2)  # You can adjust the number of components
pca_result = pca.fit_transform(combined_points)
#pca_result

# Perform UMAP with increased neighbors
reducer = umap.UMAP(n_neighbors=50, min_dist=0.2, n_components=2)
embedding_combined = reducer.fit_transform(pca_result)

# Plotting
plt.figure(figsize=(6, 4), dpi=600)
colorlist = ['#3145a8', '#40a8f7', '#f5bf36', '#fa2616']  # Green and Magenta for interpolated points
fibro_general_list = ['qFibro','iFibro','Myfibro','eFibro']
for i in range(4): # 
    type = fibro_general_list[i]
    plt.scatter(embedding_combined[combined_labels == type, 0], 
                embedding_combined[combined_labels == type, 1], 
                c=colorlist[i], 
                s=8, 
                alpha=1)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.axis('off')
#plt.title('UMAP with Linear Interpolation between day0 and day3')
plt.show()


In [ ]:


pred_tensor =np.array( torch.cat([sample_interp0,sample_interp1,sample_interp2,sample_interp3], dim=0).detach().cpu())
all_new_data = anndata.AnnData(pred_tensor)
all_new_data.obs['weights'] = combined_labels
all_new_data.obs['weights'] = all_new_data.obs['weights'].astype(str).astype('category')
all_new_data.var_names = train_adata.var_names

sc.tl.rank_genes_groups(all_new_data, groupby='weights', method='wilcoxon')
all_new_data.layers['scaled'] = sc.pp.scale(all_new_data, copy=True).X


In [ ]:
marker_genes_dict = {
    'qFibro': ['APOD', 'CFD', 'CLU','PI16'],
    'eFibro': ['COL1A1', 'COL1A2', 'COL6A3','LUM', 'DCN', 'COL3A1'],
    'iFibro': ['CXCL12',  'CCL2', 'ICAM1',  'IL33', 'CD74'],
    'Myfibro': [ "ACTA2",'MYL9','TPM2','TAGLN']
}



In [ ]:
fig,ax = plt.subplots(figsize =(9,7),dpi=800)
sc.pl.rank_genes_groups_matrixplot(all_new_data, 
                                dendrogram=False,
                                n_genes=5, 
                                   use_raw=False, 
                                swap_axes=True, 
                                # vmin=-1, vmax=1, 
                                cmap='RdBu_r', 
                               layer='scaled', 
                                   ax=ax,
                                show=False)


In [ ]:
fig,ax = plt.subplots(figsize =(10,4),dpi=300)
sc.pl.dotplot(all_new_data, 
              marker_genes_dict, 'weights', 
              vmin=0,
              vmax=1,
              dendrogram=False,
              ax=ax,
              cmap='Blues',
             )


In [ ]:
all_new_data.to_df()


In [ ]:
######################### Also try the paga method again ##################
#import scanpy.api as sc
import scanpy as sc
import numpy as np
import pandas as pd
#from scanpy.tools import rna_velocity
import loompy
import matplotlib.pyplot as pl
from anndata import AnnData
import seaborn as sns
from scipy.sparse import csr_matrix
import networkx as nx

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, transparent=True)
sc.logging.print_versions


In [ ]:
adata = sc.read_h5ad('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_BatT_12608.h5ad')


In [ ]:
# muscle_clusters = ['muscle body', 'muscle pharynx', 'muscle progenitors']
# subset_muscle = adata.obs['clusters'].isin(muscle_clusters)
# adata = adata[subset_muscle]
sc.tl.tsne(adata, n_pcs=50, random_state=42)  # 结果会自动存入 adata.obsm['X_tsne']
# sc.pl.tsne(adata, color='Celltype_general', title='RNA velocity', legend_loc='on data export')


In [ ]:
sc.pl.tsne(
    adata,
    color='Celltype_general',  # 按细胞类型着色
    title='RNA velocity',     # 图标题
    legend_loc='right margin',        # 图例位置（可选：'on data'/'right'/'none'）
    show=True              # 显示图片
    # save='tsne_celltype.png'  # 可选：保存图片到当前目录
)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import warnings
warnings.filterwarnings('ignore')

# 设置绘图风格（Cancer Cell风格）
plt.style.use('default')
rcParams['figure.figsize'] = (10, 8)
rcParams['figure.dpi'] = 150
rcParams['font.size'] = 12
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']
rcParams['axes.linewidth'] = 1.5
rcParams['axes.edgecolor'] = 'black'
rcParams['axes.labelcolor'] = 'black'
rcParams['xtick.color'] = 'black'
rcParams['ytick.color'] = 'black'
rcParams['legend.fontsize'] = 10

# 定义Cancer Cell风格的配色方案
cancer_cell_colors = {
    'hatching gland': '#E41A1C',  # 红色
    'germline': '#377EB8',         # 蓝色
    'prechordal plate': '#4DAF4A',  # 绿色
    'notochord': '#984EA3',         # 紫色
    'placentation': '#FF7F00',      # 橙色
    'ectoderm': '#FFFF33',          # 黄色
    'epiblast-like': '#A65628',     # 棕色
    'mesoderm': '#F781BF',          # 粉色
    'EML': '#999999',                # 灰色
    'lateral line': '#66C2A5',      # 青绿色
    'endoderm': '#FC8D62',           # 橙红色
    'pronephric duct': '#8DA0CB',    # 紫色
    'proctodeum': '#E78AC3',         # 粉色
    'pharyngeal': '#A6D854',         # 浅绿色
    'myotome': '#FFD92F',             # 浅黄色
    'neural crest': '#E5C494',        # 米色
    'pancreas primordiuatic': '#B3B3B3',  # 灰色
    'muscle': '#FBB4AE',              # 浅粉色
    'retina': '#B3CDE3',              # 浅蓝色
    'optic': '#CCEBC5',               # 浅绿色
    'periderm': '#DECBE4',            # 浅紫色
    'lens': '#FED9A6',                 # 浅橙色
    'early epidermal progenitors': '#E7298A',  # 深粉色
    'epidermal neoblasts': '#666666',  # 深灰色
    'late epidermal progenitors 1': '#A6761D',  # 棕色
    'late epidermal progenitors 2': '#1B9E77',  # 蓝绿色
    'epidermis': '#7570B3'  # 紫色
}



In [ ]:
def prepare_paga_analysis(adata):
    """
    准备PAGA分析所需的数据和参数
    """
    print("Step 1: 准备PAGA分析...")
    
    # 检查并确保必要的计算已完成
    if 'X_pca' not in adata.obsm:
        print("计算PCA...")
        sc.pp.pca(adata, n_comps=50, use_highly_variable=False)
    
    if 'neighbors' not in adata.uns:
        print("计算邻域图...")
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
    
    # 计算UMAP（如果还没有）
    if 'X_umap' not in adata.obsm:
        print("计算UMAP...")
        sc.tl.umap(adata)
    
    print("PAGA准备完成！")
    return adata
    

In [ ]:
def run_paga_analysis(adata, group_key='Celltype', root_cell_type=None):
    """
    运行PAGA轨迹分析
    
    Parameters:
    -----------
    adata : AnnData
        单细胞数据对象
    group_key : str
        用于分组的obs列名
    root_cell_type : str
        根细胞类型（可选）
    """
    print(f"Step 2: 运行PAGA分析，使用分组: {group_key}")
    
    # 确保分组是类别型
    if group_key not in adata.obs.columns:
        raise ValueError(f"分组列 {group_key} 不存在于adata.obs中")
    
    # 运行PAGA
    sc.tl.paga(adata, groups=group_key)
    
    # 如果指定了根细胞类型，设置PAGA根
    if root_cell_type is not None:
        if root_cell_type in adata.obs[group_key].unique():
            # 创建根细胞类型的索引
            groups = adata.obs[group_key].cat.categories
            root_idx = list(groups).index(root_cell_type)
            adata.uns['paga']['root'] = root_idx
            print(f"设置根细胞类型为: {root_cell_type}")
    
    # 计算PAGA的UMAP（基于PAGA的布局）
    sc.tl.umap(adata, init_pos='paga')
    
    print("PAGA分析完成！")
    return adata
    

In [ ]:
sc.tl.umap(adata, init_pos='random')


In [ ]:
help(sc.pl.paga)


In [ ]:
root_cell_type ='qFibro'
if root_cell_type is not None:
    if root_cell_type in adata.obs[group_key].unique():
        # 创建根细胞类型的索引
        groups = adata.obs[group_key].cat.categories
        root_idx = list(groups).index(root_cell_type)
        adata.uns['paga']['root'] = root_idx
        print(f"设置根细胞类型为: {root_cell_type}")


In [ ]:
# help(sc.pl.paga)

In [ ]:
sc.pl.paga(
    adata,
    layout='fr',
    threshold=0.015,
    root=None,
    node_size_scale=0.5,
    node_size_power=0.9,
    edge_width_scale=1,
    max_edge_width=0.7,
    fontsize=3.5)


In [ ]:
adata_coarse = adata.copy()


In [ ]:
sc.tl.paga(adata_coarse, groups='Celltype')


In [ ]:
sc.pl.paga(
    adata_coarse,
    layout='fr',
    threshold=0.01,
    fontsize=5,
    node_size_scale=0.5,
    node_size_power=0.5,
    max_edge_width=0.7,
    title='Celltype paga',
   # text_kwds={'fontweight': 'bold'},
    frameon=False)


In [ ]:
help(sc.pl.paga)


In [ ]:
import scanpy.api as sc
import numpy as np
import pandas as pd
from scanpy.tools import rna_velocity
import loompy
import matplotlib.pyplot as pl
from anndata import AnnData
import seaborn as sns
from scipy.sparse import csr_matrix
import networkx as nx

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, transparent=True)
sc.logging.print_versions()

In [ ]:
adata


In [ ]:


def plot_paga_graph(adata, group_key='Celltype', colors=None, save_path=None):
    """
    绘制PAGA图（带箭头的轨迹图）
    """
    print("Step 3: 绘制PAGA图...")
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # 使用自定义颜色或默认颜色
    if colors is None:
        colors = cancer_cell_colors
    
    # 绘制PAGA图
    sc.pl.paga(
        adata,
        color=group_key,
        title='PAGA Trajectory Analysis',
        fontsize=12,
        node_size_scale=3,
        edge_width_scale=1.5,
        min_edge_width=0.5,
        max_edge_width=4,
        show=False,
        ax=ax,
        frameon=True,
        layout='fa',  # Force-directed layout
        threshold=0.01  # 只显示概率>0.01的连接
    )
    
    # 美化图形
    ax.set_title('PAGA Trajectory Analysis', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('PAGA Dimension 1', fontsize=14, fontweight='bold')
    ax.set_ylabel('PAGA Dimension 2', fontsize=14, fontweight='bold')
    
    # 添加网格线（Cancer Cell风格）
    ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)
    
    # 设置图例位置
    if ax.get_legend():
        ax.get_legend().set_bbox_to_anchor((1.05, 1))
        ax.get_legend().set_title(group_key, prop={'size': 12, 'weight': 'bold'})
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"图形已保存到: {save_path}")
    
    plt.show()
    return fig

def plot_paga_with_umap(adata, group_key='Celltype', colors=None, save_path=None):
    """
    在UMAP投影上绘制PAGA轨迹
    """
    print("Step 4: 在UMAP上绘制PAGA轨迹...")
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    if colors is None:
        colors = cancer_cell_colors
    
    # 子图1: 带轨迹的UMAP
    sc.pl.paga_compare(
        adata,
        basis='umap',
        color=group_key,
        title='UMAP with PAGA Trajectories',
        fontsize=12,
        node_size_scale=3,
        edge_width_scale=1.5,
        min_edge_width=0.5,
        max_edge_width=4,
        show=False,
        ax=axes[0],
        frameon=True,
        threshold=0.01,
        legend_loc='on data'  # 图例放在数据点上
    )
    
    # 子图2: 纯UMAP着色
    sc.pl.umap(
        adata,
        color=group_key,
        title='UMAP by Cell Types',
        show=False,
        ax=axes[1],
        palette=colors,
        legend_loc='right margin',
        frameon=True,
        s=20  # 点大小
    )
    
    # 美化图形
    for ax in axes:
        ax.set_xlabel('UMAP1', fontsize=14, fontweight='bold')
        ax.set_ylabel('UMAP2', fontsize=14, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)
    
    axes[0].set_title('UMAP with PAGA Trajectories', fontsize=16, fontweight='bold')
    axes[1].set_title('UMAP by Cell Types', fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"图形已保存到: {save_path}")
    
    plt.show()
    return fig

def plot_celltype_specific_trajectory(adata, group_key='Celltype', 
                                      celltypes_of_interest=None, 
                                      colors=None, save_path=None):
    """
    绘制特定细胞类型的轨迹
    """
    print("Step 5: 绘制特定细胞类型轨迹...")
    
    if celltypes_of_interest is None:
        # 默认为表皮相关的细胞类型
        celltypes_of_interest = [
            'early epidermal progenitors',
            'epidermal neoblasts',
            'late epidermal progenitors 1',
            'late epidermal progenitors 2',
            'epidermis'
        ]
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    if colors is None:
        colors = cancer_cell_colors
    
    # 创建颜色字典（只包含感兴趣的细胞类型）
    filtered_colors = {k: v for k, v in colors.items() 
                      if k in celltypes_of_interest}
    
    # 绘制PAGA图（只显示感兴趣的细胞类型）
    sc.pl.paga(
        adata,
        color=group_key,
        groups=celltypes_of_interest,
        title='Cell Type Specific Trajectory Analysis',
        fontsize=12,
        node_size_scale=4,
        edge_width_scale=2,
        min_edge_width=0.5,
        max_edge_width=5,
        show=False,
        ax=ax,
        frameon=True,
        layout='fa',
        threshold=0.01
    )
    
    # 美化图形
    ax.set_title('Epidermal Lineage Trajectory', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('PAGA Dimension 1', fontsize=14, fontweight='bold')
    ax.set_ylabel('PAGA Dimension 2', fontsize=14, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)
    
    if ax.get_legend():
        ax.get_legend().set_bbox_to_anchor((1.05, 1))
        ax.get_legend().set_title('Epidermal Lineage', prop={'size': 12, 'weight': 'bold'})
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"图形已保存到: {save_path}")
    
    plt.show()
    return fig

def plot_paga_heatmap(adata, group_key='Celltype', save_path=None):
    """
    绘制PAGA连接矩阵热图
    """
    print("Step 6: 绘制PAGA连接热图...")
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # 获取PAGA连接矩阵
    if 'paga' not in adata.uns or 'connectivities' not in adata.uns['paga']:
        print("重新计算PAGA...")
        sc.tl.paga(adata, groups=group_key)
    
    connectivities = adata.uns['paga']['connectivities'].toarray()
    
    # 获取细胞类型标签
    celltypes = adata.obs[group_key].cat.categories
    
    # 创建热图数据框
    heatmap_df = pd.DataFrame(
        connectivities,
        index=celltypes,
        columns=celltypes
    )
    
    # 绘制热图
    sns.heatmap(
        heatmap_df,
        annot=True,
        fmt='.2f',
        cmap='Reds',
        cbar_kws={'label': 'Transition Probability'},
        square=True,
        linewidths=1,
        linecolor='white',
        ax=ax,
        annot_kws={'size': 8}
    )
    
    # 美化热图
    ax.set_title('PAGA Transition Probabilities Between Cell Types', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Target Cell Types', fontsize=14, fontweight='bold')
    ax.set_ylabel('Source Cell Types', fontsize=14, fontweight='bold')
    
    # 旋转标签以便阅读
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"热图已保存到: {save_path}")
    
    plt.show()
    return fig

def extract_transition_probabilities(adata, group_key='Celltype', threshold=0.01):
    """
    提取并保存过渡概率矩阵
    """
    print(f"Step 7: 提取过渡概率（阈值 > {threshold}）...")
    
    if 'paga' not in adata.uns or 'connectivities' not in adata.uns['paga']:
        print("需要先运行PAGA分析")
        return None
    
    # 获取连接矩阵
    connectivities = adata.uns['paga']['connectivities'].toarray()
    celltypes = adata.obs[group_key].cat.categories
    
    # 创建数据框
    prob_df = pd.DataFrame(
        connectivities,
        index=celltypes,
        columns=celltypes
    )
    
    # 只保留大于阈值的连接
    prob_df_filtered = prob_df[prob_df > threshold]
    
    # 转换为长格式
    prob_long = prob_df_filtered.stack().reset_index()
    prob_long.columns = ['Source', 'Target', 'Probability']
    prob_long = prob_long[prob_long['Probability'] > threshold]
    prob_long = prob_long.sort_values('Probability', ascending=False)
    
    print(f"找到 {len(prob_long)} 个显著连接（概率 > {threshold}）")
    print("\nTop 10 过渡概率:")
    print(prob_long.head(10))
    
    return prob_long

def create_comprehensive_trajectory_figure(adata, group_key='Celltype', 
                                           colors=None, save_path=None):
    """
    创建包含多个子图的综合轨迹分析图
    """
    print("Step 8: 创建综合轨迹分析图...")
    
    if colors is None:
        colors = cancer_cell_colors
    
    # 创建4个子图
    fig = plt.figure(figsize=(24, 20))
    
    # 定义子图网格
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    
    # 子图1: PAGA图
    ax1 = fig.add_subplot(gs[0, 0])
    sc.pl.paga(
        adata,
        color=group_key,
        title='PAGA Trajectory Network',
        fontsize=12,
        node_size_scale=3,
        edge_width_scale=1.5,
        show=False,
        ax=ax1,
        palette=colors,
        add_edge_weights=True,
        threshold=0.01,
        frameon=True
    )
    ax1.set_title('A. PAGA Trajectory Network', fontsize=16, fontweight='bold', loc='left')
    
    # 子图2: UMAP with PAGA
    ax2 = fig.add_subplot(gs[0, 1])
    sc.pl.paga_compare(
        adata,
        basis='umap',
        color=group_key,
        title='UMAP with Trajectories',
        fontsize=12,
        node_size_scale=3,
        edge_width_scale=1.5,
        show=False,
        ax=ax2,
        palette=colors,
        add_edge_weights=True,
        threshold=0.01,
        legend_loc='on data'
    )
    ax2.set_title('B. UMAP with PAGA Trajectories', fontsize=16, fontweight='bold', loc='left')
    
    # 子图3: 特定细胞类型轨迹
    ax3 = fig.add_subplot(gs[1, 0])
    epidermal_cells = ['early epidermal progenitors', 'epidermal neoblasts', 
                       'late epidermal progenitors 1', 'late epidermal progenitors 2', 
                       'epidermis']
    filtered_colors = {k: v for k, v in colors.items() if k in epidermal_cells}
    sc.pl.paga(
        adata,
        color=group_key,
        groups=epidermal_cells,
        title='Epidermal Lineage',
        fontsize=12,
        node_size_scale=4,
        edge_width_scale=2,
        show=False,
        ax=ax3,
        palette=filtered_colors,
        add_edge_weights=True,
        threshold=0.01,
        frameon=True
    )
    ax3.set_title('C. Epidermal Lineage Trajectory', fontsize=16, fontweight='bold', loc='left')
    
    # 子图4: 热图
    ax4 = fig.add_subplot(gs[1, 1])
    connectivities = adata.uns['paga']['connectivities'].toarray()
    celltypes = adata.obs[group_key].cat.categories
    sns.heatmap(
        connectivities,
        annot=True,
        fmt='.2f',
        cmap='Reds',
        cbar_kws={'label': 'Transition Probability'},
        square=True,
        linewidths=1,
        linecolor='white',
        ax=ax4,
        xticklabels=celltypes,
        yticklabels=celltypes,
        annot_kws={'size': 7}
    )
    ax4.set_title('D. Transition Probability Matrix', fontsize=16, fontweight='bold', loc='left')
    ax4.set_xlabel('Target Cell Types', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Source Cell Types', fontsize=12, fontweight='bold')
    plt.setp(ax4.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    plt.setp(ax4.get_yticklabels(), rotation=0, fontsize=8)
    
    plt.suptitle('Single-Cell Trajectory Analysis by PAGA', 
                 fontsize=20, fontweight='bold', y=0.98)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"综合图形已保存到: {save_path}")
    
    plt.show()
    return fig




In [ ]:
# plot_paga_with_umap(adata, group_key=group_key, colors=cancer_cell_colors )


In [ ]:
# 主执行函数
def main_trajectory_analysis(adata, group_key='Celltype', root_cell_type=None):
    """
    主函数：执行完整的PAGA轨迹分析流程
    """
    print("="*60)
    print("开始PAGA轨迹分析")
    print("="*60)
    
    # 检查数据
    print(f"数据维度: {adata.shape}")
    print(f"细胞类型数量: {len(adata.obs[group_key].unique())}")
    
    # Step 1: 准备数据
    adata = prepare_paga_analysis(adata)
    
    # Step 2: 运行PAGA
    adata = run_paga_analysis(adata, group_key=group_key, root_cell_type=root_cell_type)
    
    # Step 3-6: 绘制各种图形
    # 3.1 PAGA图
    fig1 = plot_paga_graph(adata, group_key=group_key, colors=cancer_cell_colors,
                          save_path='paga_trajectory.png')
    
    # 3.2 UMAP with PAGA
    fig2 = plot_paga_with_umap(adata, group_key=group_key, colors=cancer_cell_colors,
                               save_path='umap_with_paga.png')
    
    # 3.3 特定细胞类型轨迹
    fig3 = plot_celltype_specific_trajectory(adata, group_key=group_key, 
                                             colors=cancer_cell_colors,
                                             save_path='epidermal_trajectory.png')
    
    # 3.4 热图
    fig4 = plot_paga_heatmap(adata, group_key=group_key,
                             save_path='paga_heatmap.png')
    
    # 3.5 综合图形
    fig5 = create_comprehensive_trajectory_figure(adata, group_key=group_key,
                                                  colors=cancer_cell_colors,
                                                  save_path='comprehensive_trajectory.png')
    
    # 提取过渡概率
    transitions = extract_transition_probabilities(adata, group_key=group_key, threshold=0.01)
    
    # 保存过渡概率到CSV
    if transitions is not None:
        transitions.to_csv('transition_probabilities.csv', index=False)
        print("过渡概率已保存到: transition_probabilities.csv")
    
    print("\n" + "="*60)
    print("PAGA轨迹分析完成！")
    print("="*60)
    
    return adata


In [ ]:
# adata = main_trajectory_analysis(adata, group_key='Celltype_general')

In [ ]:
adata = prepare_paga_analysis(adata)

# Step 2: 运行PAGA
# adata = run_paga_analysis(adata, group_key='Celltype_general', root_cell_type='qFibro')



In [ ]:
# Step 3-6: 绘制各种图形
# 3.1 PAGA图
fig1 = plot_paga_graph(adata, group_key=group_key, colors=cancer_cell_colors,
                      # save_path='paga_trajectory.png')

# 3.2 UMAP with PAGA
fig2 = plot_paga_with_umap(adata, group_key=group_key, colors=cancer_cell_colors,
                           save_path='umap_with_paga.png')

# 3.3 特定细胞类型轨迹
fig3 = plot_celltype_specific_trajectory(adata, group_key=group_key, 
                                         colors=cancer_cell_colors,
                                         save_path='epidermal_trajectory.png')

# 3.4 热图
fig4 = plot_paga_heatmap(adata, group_key=group_key,
                         save_path='paga_heatmap.png')

# 3.5 综合图形
fig5 = create_comprehensive_trajectory_figure(adata, group_key=group_key,
                                              colors=cancer_cell_colors,
                                              save_path='comprehensive_trajectory.png')

# 提取过渡概率
transitions = extract_transition_probabilities(adata, group_key=group_key, threshold=0.01)

# 保存过渡概率到CSV
if transitions is not None:
    transitions.to_csv('transition_probabilities.csv', index=False)
    print("过渡概率已保存到: transition_probabilities.csv")

print("\n" + "="*60)
print("PAGA轨迹分析完成！")
print("="*60)
